## CÉLULA 0 – IMPORTS E CONFIGURAÇÃO GLOBAL

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap, Normalize
import matplotlib.cm as cm
import seaborn as sns
from scipy import stats
from math import radians
from numpy import sin, cos, sqrt, arctan2

# Paleta
C_SUDESTE = "#2C3E50"
C_SUL     = "#3498DB"
C_NE      = "#E67E22"
C_NORTE   = "#27AE60"
C_CO      = "#8E44AD"
ACCENT    = "#E74C3C"
GOLD      = "#F39C12"

REGION_COLORS = {
    "Sudeste":     C_SUDESTE,
    "Sul":         C_SUL,
    "Nordeste":    C_NE,
    "Norte":       C_NORTE,
    "Centro-Oeste":C_CO,
}

STATE_REGION = {
    "SP":"Sudeste","RJ":"Sudeste","MG":"Sudeste","ES":"Sudeste",
    "PR":"Sul","SC":"Sul","RS":"Sul",
    "BA":"Nordeste","PE":"Nordeste","CE":"Nordeste","MA":"Nordeste",
    "PI":"Nordeste","RN":"Nordeste","PB":"Nordeste","AL":"Nordeste","SE":"Nordeste",
    "PA":"Norte","AM":"Norte","AC":"Norte","RO":"Norte","RR":"Norte","AP":"Norte","TO":"Norte",
    "DF":"Centro-Oeste","GO":"Centro-Oeste","MT":"Centro-Oeste","MS":"Centro-Oeste",
}

plt.rcParams.update({
    "figure.facecolor":  "white",
    "axes.facecolor":    "#F8F9FA",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.4,
    "grid.color":        "#DEE2E6",
    "font.family":       "DejaVu Sans",
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "axes.labelsize":    11,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
})

DATA_PATH = "/content/olist_dataset"

def salvar(nome, dpi=150):
    plt.tight_layout()
    plt.savefig(f"{nome}.png", dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"✔  Salvo → {nome}.png\n")

def titulo_relatorio(texto_principal, subtitulo=""):
    """Estilo de título padronizado para cada gráfico."""
    plt.suptitle(texto_principal, fontsize=15, fontweight="bold", y=1.02)
    if subtitulo:
        plt.title(subtitulo, fontsize=10, color="#6C757D", pad=8)

## CÉLULA 1 – CARREGAMENTO DOS DADOS

In [ ]:
print("Carregando dados…")

orders      = pd.read_csv(DATA_PATH + "/olist_orders_dataset.csv",
                          parse_dates=["order_purchase_timestamp",
                                       "order_approved_at",
                                       "order_delivered_carrier_date",
                                       "order_delivered_customer_date",
                                       "order_estimated_delivery_date"])
customers   = pd.read_csv(DATA_PATH + "/olist_customers_dataset.csv")
items       = pd.read_csv(DATA_PATH + "/olist_order_items_dataset.csv")
payments    = pd.read_csv(DATA_PATH + "/olist_order_payments_dataset.csv")
reviews     = pd.read_csv(DATA_PATH + "/olist_order_reviews_dataset.csv",
                          parse_dates=["review_creation_date"])
products    = pd.read_csv(DATA_PATH + "/olist_products_dataset.csv")
sellers     = pd.read_csv(DATA_PATH + "/olist_sellers_dataset.csv")
geo         = pd.read_csv(DATA_PATH + "/olist_geolocation_dataset.csv")
cat_transl  = pd.read_csv(DATA_PATH + "/product_category_name_translation.csv")

cat_map = cat_transl.set_index("product_category_name")["product_category_name_english"].to_dict()
products["category_en"] = products["product_category_name"].map(cat_map)

# Geo médio por CEP
geo_mean = (geo.groupby("geolocation_zip_code_prefix")
            .agg(lat=("geolocation_lat","mean"), lng=("geolocation_lng","mean"))
            .reset_index())

# Joins base
orders_full = (orders
               .merge(customers[["customer_id","customer_unique_id",
                                  "customer_state","customer_zip_code_prefix"]],
                      on="customer_id", how="left"))
orders_full["year"]  = orders_full["order_purchase_timestamp"].dt.year
orders_full["month"] = orders_full["order_purchase_timestamp"].dt.month
orders_full["region"] = orders_full["customer_state"].map(STATE_REGION)

items_full = (items
              .merge(products[["product_id","product_category_name","category_en"]],
                     on="product_id", how="left")
              .merge(orders_full[["order_id","customer_state","region","year","month",
                                   "customer_unique_id","order_status"]],
                     on="order_id", how="left"))

print("✔  Dados carregados!")

## SEÇÃO 1 – COMPORTAMENTO DOS USUÁRIOS

### DISTRIBUIÇÃO REGIONAL DE CLIENTES

In [ ]:
print("Gerando Gráfico 1 – Distribuição Regional…")

cust_state = (customers.groupby("customer_state")
              .size().reset_index(name="clientes"))
cust_state["region"] = cust_state["customer_state"].map(STATE_REGION)
cust_state = cust_state.sort_values("clientes", ascending=True)

region_total = cust_state.groupby("region")["clientes"].sum().reset_index()
region_total["pct"] = region_total["clientes"] / region_total["clientes"].sum() * 100
region_total = region_total.sort_values("clientes", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 9),
                          gridspec_kw={"width_ratios": [2, 1]})
fig.suptitle("Distribuição de Clientes por Estado e Região", fontsize=16,
             fontweight="bold", y=1.01)

# Barra horizontal – por estado
ax = axes[0]
bar_colors = [REGION_COLORS.get(r, "#95A5A6")
              for r in cust_state["region"]]
bars = ax.barh(cust_state["customer_state"], cust_state["clientes"],
               color=bar_colors, edgecolor="white", height=0.7)
for bar in bars:
    v = bar.get_width()
    ax.text(v + 120, bar.get_y() + bar.get_height()/2,
            f"{v:,.0f}", va="center", fontsize=8.5, color="#333")
ax.set_xlabel("Número de Clientes")
ax.set_title("Clientes Cadastrados por Estado\n(ordenado por volume)")
# Legenda de regiões
legend_patches = [mpatches.Patch(color=c, label=r)
                  for r, c in REGION_COLORS.items()]
ax.legend(handles=legend_patches, title="Região", loc="lower right",
          fontsize=8.5, title_fontsize=9)

# Donut – por região
ax2 = axes[1]
wedge_colors = [REGION_COLORS.get(r, "#95A5A6") for r in region_total["region"]]
wedges, texts, autotexts = ax2.pie(
    region_total["clientes"],
    labels=None,
    colors=wedge_colors,
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.78,
    wedgeprops=dict(width=0.52, edgecolor="white", linewidth=2)
)
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight("bold")
    at.set_color("white")
ax2.legend(region_total["region"], title="Região",
           loc="lower center", bbox_to_anchor=(0.5, -0.12), fontsize=9, ncol=2)
ax2.set_title(f"Share por Região\n(Total: {cust_state['clientes'].sum():,} clientes)")

# Anotação da média global
total_geral = cust_state["clientes"].sum()
ax.axvline(total_geral / len(cust_state), color=ACCENT, linestyle="--",
           linewidth=1.2, alpha=0.7)
ax.text(total_geral / len(cust_state) + 50, 1,
        f"Média: {total_geral/len(cust_state):,.0f}", color=ACCENT,
        fontsize=8, va="bottom")

salvar("g01_distribuicao_regional_clientes")

### DISTÂNCIA MÉDIA DE FRETE POR ESTADO (DESTINO)

In [ ]:
print("Gerando Gráfico 2 – Distância Média de Frete…")

def haversine_np(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = [np.radians(x) for x in [lat1, lon1, lat2, lon2]]
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1-a))

# Construir tabela de distâncias
dist_df = (items[["order_id","seller_id"]]
           .merge(sellers[["seller_id","seller_zip_code_prefix"]], on="seller_id", how="left")
           .merge(geo_mean.rename(columns={"geolocation_zip_code_prefix":"seller_zip_code_prefix",
                                            "lat":"seller_lat","lng":"seller_lng"}),
                  on="seller_zip_code_prefix", how="left")
           .merge(orders_full[["order_id","customer_zip_code_prefix","customer_state"]],
                  on="order_id", how="left")
           .merge(geo_mean.rename(columns={"geolocation_zip_code_prefix":"customer_zip_code_prefix",
                                            "lat":"cust_lat","lng":"cust_lng"}),
                  on="customer_zip_code_prefix", how="left")
           .dropna(subset=["seller_lat","cust_lat"]))

dist_df["distance_km"] = haversine_np(
    dist_df["seller_lat"], dist_df["seller_lng"],
    dist_df["cust_lat"],   dist_df["cust_lng"])

state_dist = (dist_df.groupby("customer_state")["distance_km"]
              .mean().reset_index(name="avg_km")
              .sort_values("avg_km", ascending=True))
state_dist["region"] = state_dist["customer_state"].map(STATE_REGION)
media_global = dist_df["distance_km"].mean()

fig, ax = plt.subplots(figsize=(14, 10))
fig.suptitle("Distância Média Percorrida pelas Cargas por Estado de Destino",
             fontsize=15, fontweight="bold", y=1.01)

# Gradiente de cor: menor distância = verde, maior = vermelho
norm  = Normalize(state_dist["avg_km"].min(), state_dist["avg_km"].max())
cmap  = LinearSegmentedColormap.from_list("dist", ["#27AE60","#F39C12","#E74C3C"])
colors_dist = [cmap(norm(v)) for v in state_dist["avg_km"]]

bars = ax.barh(state_dist["customer_state"], state_dist["avg_km"],
               color=colors_dist, edgecolor="white", height=0.75)
ax.axvline(media_global, color="#2C3E50", linestyle="--", linewidth=1.5,
           label=f"Média global: {media_global:,.0f} km")
for bar in bars:
    v = bar.get_width()
    ax.text(v + 15, bar.get_y() + bar.get_height()/2,
            f"{v:,.0f} km", va="center", fontsize=8.5)
ax.set_xlabel("Distância Média (km)")
ax.set_title("Todos os estados recebem cargas de distâncias semelhantes (1.600–3.400 km)\n"
             "→ logística não explica concentração regional de clientes", fontsize=10, color="#555")
ax.legend(fontsize=10)

# Colorbar
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.4, pad=0.01)
cbar.set_label("Distância (km)", fontsize=9)

salvar("g02_distancia_media_frete_estado")

### RECOMPRA POR ESTADO

In [ ]:
print("Gerando Gráfico 3 – Recompra por Estado…")

purchase_counts = (orders_full[orders_full["order_status"] == "delivered"]
                   .groupby("customer_unique_id")
                   .agg(n_compras=("order_id","count"),
                        customer_state=("customer_state","first"))
                   .reset_index())

recompra = (purchase_counts.groupby("customer_state")
            .agg(avg_compras=("n_compras","mean"),
                 pct_recompra=("n_compras", lambda x: (x > 1).mean() * 100),
                 n_clientes=("customer_unique_id","count"))
            .reset_index()
            .sort_values("avg_compras", ascending=True))
recompra["region"] = recompra["customer_state"].map(STATE_REGION)
media_recompra = purchase_counts["n_compras"].mean()

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
fig.suptitle("Comportamento de Recompra por Estado", fontsize=16,
             fontweight="bold", y=1.01)

# Média de compras por cliente
ax = axes[0]
bar_colors_r = [REGION_COLORS.get(r,"#95A5A6") for r in recompra["region"]]
bars = ax.barh(recompra["customer_state"], recompra["avg_compras"],
               color=bar_colors_r, edgecolor="white", height=0.72)
ax.axvline(media_recompra, color=ACCENT, linestyle="--", linewidth=1.5,
           label=f"Média: {media_recompra:.2f} compras/cliente")
ax.axvline(1, color="#ADB5BD", linestyle=":", linewidth=1.2,
           label="Linha de 1 compra (sem recompra)")
for bar in bars:
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f"{bar.get_width():.2f}", va="center", fontsize=8.5)
ax.set_xlabel("Média de Compras por Cliente")
ax.set_title("Média de Compras por Cliente por Estado\n"
             "(acima de 1.0 = há recompra)")
ax.legend(fontsize=9)

# Porcentagem de clientes que recompraram
ax = axes[1]
recompra_s = recompra.sort_values("pct_recompra", ascending=True)
bar_colors_p = [REGION_COLORS.get(r,"#95A5A6") for r in recompra_s["region"]]
bars2 = ax.barh(recompra_s["customer_state"], recompra_s["pct_recompra"],
                color=bar_colors_p, edgecolor="white", height=0.72)
media_pct = recompra_s["pct_recompra"].mean()
ax.axvline(media_pct, color=ACCENT, linestyle="--", linewidth=1.5,
           label=f"Média: {media_pct:.1f}%")
for bar in bars2:
    v = bar.get_width()
    ax.text(v + 0.1, bar.get_y() + bar.get_height()/2,
            f"{v:.1f}%", va="center", fontsize=8.5)
ax.set_xlabel("% de Clientes que Fizeram Mais de 1 Compra")
ax.set_title("Taxa de Recompra por Estado\n"
             "(% clientes com 2+ pedidos entregues)")
legend_patches = [mpatches.Patch(color=c, label=r)
                  for r, c in REGION_COLORS.items()]
ax.legend(handles=legend_patches, title="Região", fontsize=8, title_fontsize=9)

salvar("g03_recompra_por_estado")

### DISPERSÃO DO TICKET MÉDIO

In [ ]:
print("Gerando Gráfico 4 – Distribuição de Ticket…")

ticket = (items.groupby("order_id")["price"].sum().reset_index(name="ticket"))
ticket = ticket[ticket["ticket"] < ticket["ticket"].quantile(0.99)]  # remove outliers extremos
media_tk = ticket["ticket"].mean()
mediana_tk = ticket["ticket"].median()
p75 = ticket["ticket"].quantile(0.75)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle("Distribuição do Ticket por Pedido", fontsize=15,
             fontweight="bold", y=1.01)

# Histograma com KDE
ax = axes[0]
ax.hist(ticket["ticket"], bins=80, color=C_SUDESTE, alpha=0.6,
        edgecolor="white", density=True, label="Frequência")
kde_x = np.linspace(0, ticket["ticket"].max(), 500)
kde = stats.gaussian_kde(ticket["ticket"])
ax.plot(kde_x, kde(kde_x), color=ACCENT, linewidth=2.5, label="Curva KDE")
ax.axvline(media_tk,   color=GOLD,     linestyle="--", lw=1.8,
           label=f"Média: R$ {media_tk:.0f}")
ax.axvline(mediana_tk, color=C_NORTE,  linestyle="--", lw=1.8,
           label=f"Mediana: R$ {mediana_tk:.0f}")
ax.axvline(p75,        color=C_NE,     linestyle=":",  lw=1.5,
           label=f"P75: R$ {p75:.0f}")
ax.set_xlabel("Valor do Pedido (R$)")
ax.set_ylabel("Densidade")
ax.set_title("Histograma + Densidade\n"
             "Alta concentração em pedidos de baixo valor")
ax.set_xlim(0, ticket["ticket"].quantile(0.96))
ax.legend(fontsize=9)

# Faixas de valor – % acumulada
ax2 = axes[1]
faixas = [0, 50, 100, 150, 200, 300, 500, 1000, 9999]
labels_f = ["R$ 0–50","R$ 50–100","R$ 100–150","R$ 150–200",
            "R$ 200–300","R$ 300–500","R$ 500–1k","> R$ 1.000"]
ticket["faixa"] = pd.cut(ticket["ticket"], bins=faixas, labels=labels_f)
faixa_counts = ticket["faixa"].value_counts().reindex(labels_f)
faixa_pct    = faixa_counts / faixa_counts.sum() * 100

bar_colors_f = [C_NORTE if i < 4 else (GOLD if i < 6 else ACCENT)
                for i in range(len(labels_f))]
bars = ax2.bar(labels_f, faixa_pct, color=bar_colors_f, edgecolor="white")
for bar in bars:
    v = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.3,
             f"{v:.1f}%", ha="center", va="bottom", fontsize=9)
ax2.set_xlabel("Faixa de Valor do Pedido")
ax2.set_ylabel("% dos Pedidos")
ax2.set_title(f"Concentração por Faixa de Preço\n"
              f"({faixa_pct[:4].sum():.0f}% dos pedidos custam até R$ 200)")
plt.xticks(rotation=35, ha="right")

# Legenda de cores
p1 = mpatches.Patch(color=C_NORTE, label="Baixo valor (< R$ 200)")
p2 = mpatches.Patch(color=GOLD,    label="Médio valor (R$ 200–500)")
p3 = mpatches.Patch(color=ACCENT,  label="Alto valor (> R$ 500)")
ax2.legend(handles=[p1,p2,p3], fontsize=9)

salvar("g04_distribuicao_ticket_medio")

### CATEGORIAS MAIS PEDIDAS (VOLUME)

In [ ]:
print("Gerando Gráfico 5 – Categorias Mais Pedidas…")

cat_vol = (items_full[items_full["order_status"] == "delivered"]
           .groupby("category_en")["order_id"].nunique()
           .reset_index(name="n_pedidos")
           .dropna()
           .sort_values("n_pedidos", ascending=True)
           .tail(20))

total_ped = cat_vol["n_pedidos"].sum()
cat_vol["pct"] = cat_vol["n_pedidos"] / items_full["order_id"].nunique() * 100

# Gradiente por volume
norm_v = Normalize(cat_vol["n_pedidos"].min(), cat_vol["n_pedidos"].max())
cmap_v = LinearSegmentedColormap.from_list("vol", ["#AED6F1","#1A5276"])
colors_v = [cmap_v(norm_v(v)) for v in cat_vol["n_pedidos"]]

fig, ax = plt.subplots(figsize=(14, 10))
fig.suptitle("Top 20 Categorias por Volume de Pedidos", fontsize=15,
             fontweight="bold", y=1.01)

bars = ax.barh(cat_vol["category_en"], cat_vol["n_pedidos"],
               color=colors_v, edgecolor="white", height=0.72)
for bar, (_, row) in zip(bars, cat_vol.iterrows()):
    ax.text(bar.get_width() + 60,
            bar.get_y() + bar.get_height()/2,
            f"{row['n_pedidos']:,}  ({row['pct']:.1f}%)",
            va="center", fontsize=8.5)
ax.set_xlabel("Número de Pedidos Entregues")
ax.set_title("Cama/Banho, Beleza e Esporte lideram o volume\n"
             "→ produtos de consumo recorrente dominam a plataforma",
             fontsize=10, color="#555")
ax.set_xlim(0, cat_vol["n_pedidos"].max() * 1.28)

salvar("g05_categorias_mais_pedidas")

### PEDIDOS E FATURAMENTO MENSAL (TENDÊNCIA ANUAL)

In [ ]:
print("Gerando Gráfico 6 – Pedidos e Faturamento Mensal…")

monthly_pay = (payments.merge(orders_full[["order_id","year","month","order_status"]],
                               on="order_id", how="left")
               .query("order_status == 'delivered'")
               .groupby(["year","month"])
               .agg(faturamento=("payment_value","sum"),
                    n_pedidos=("order_id","nunique"))
               .reset_index())
monthly_pay["periodo"] = pd.to_datetime(
    monthly_pay.apply(lambda r: f"{int(r['year'])}-{int(r['month']):02d}", axis=1))
monthly_pay = monthly_pay.sort_values("periodo")
# Remove meses extremos
monthly_pay = monthly_pay[monthly_pay["n_pedidos"] > 50]

fig, ax1 = plt.subplots(figsize=(16, 6))
fig.suptitle("Evolução Mensal de Pedidos e Faturamento (2016–2018)",
             fontsize=15, fontweight="bold", y=1.01)

# Barras = faturamento
ax1.bar(range(len(monthly_pay)), monthly_pay["faturamento"] / 1e3,
        color=C_SUDESTE, alpha=0.35, width=0.8, label="Faturamento (R$ mil)")
ax1.set_ylabel("Faturamento (R$ mil)", color=C_SUDESTE, fontsize=11)
ax1.tick_params(axis="y", labelcolor=C_SUDESTE)

# Linha = pedidos
ax2 = ax1.twinx()
ax2.plot(range(len(monthly_pay)), monthly_pay["n_pedidos"],
         "o-", color=ACCENT, linewidth=2.5, markersize=6, label="Nº de Pedidos")
ax2.set_ylabel("Número de Pedidos", color=ACCENT, fontsize=11)
ax2.tick_params(axis="y", labelcolor=ACCENT)

# Eixo X – labels de mês/ano
step = max(1, len(monthly_pay)//10)
ax1.set_xticks(range(0, len(monthly_pay), step))
ax1.set_xticklabels(
    [monthly_pay["periodo"].iloc[i].strftime("%b/%Y")
     for i in range(0, len(monthly_pay), step)],
    rotation=40, ha="right")

# Destaque: picos de sazonalidade
meses_pico = monthly_pay[monthly_pay["month"].isin([3, 5, 11])].index.tolist()
for idx_orig in meses_pico:
    pos = monthly_pay.index.get_loc(idx_orig)
    ax2.axvline(pos, color=GOLD, linewidth=1.0, linestyle="--", alpha=0.6)

# Legenda unificada
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1+h2, l1+l2, loc="upper left", fontsize=9)
pico_patch = mpatches.Patch(color=GOLD, alpha=0.6, label="Meses de pico (Mar/Mai/Nov)")
ax1.legend(h1+h2+[pico_patch], l1+l2+["Meses de pico"], loc="upper left", fontsize=9)

ax1.set_title("Crescimento consistente com picos em Março, Maio e Novembro\n"
              "→ campanhas promocionais nesses meses maximizam ROI",
              fontsize=10, color="#555")

salvar("g06_pedidos_faturamento_mensal")

### SAZONALIDADE MENSAL (COMPARATIVO POR ANO)

In [ ]:
print("Gerando Gráfico 7 – Sazonalidade Mensal…")

sazon = (orders_full[orders_full["order_status"] == "delivered"]
         .groupby(["year","month"])["order_id"].count()
         .reset_index(name="pedidos"))
sazon = sazon[sazon["year"].isin([2017, 2018])]  # 2016 incompleto

meses_pt = ["Jan","Fev","Mar","Abr","Mai","Jun",
            "Jul","Ago","Set","Out","Nov","Dez"]

fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle("Sazonalidade de Pedidos por Mês (Comparativo Anual)",
             fontsize=15, fontweight="bold", y=1.01)

colors_year = {2017: C_SUDESTE, 2018: ACCENT}
for year, grp in sazon.groupby("year"):
    grp_sorted = grp.sort_values("month")
    ax.plot(grp_sorted["month"], grp_sorted["pedidos"],
            "o-", color=colors_year[year], linewidth=2.5,
            markersize=7, label=str(year))
    ax.fill_between(grp_sorted["month"], grp_sorted["pedidos"],
                    alpha=0.08, color=colors_year[year])

# Anotações de pico
for month_pico, label_pico in [(3,"Março"), (5,"Maio"), (11,"Novembro")]:
    max_val = sazon[sazon["month"] == month_pico]["pedidos"].max()
    ax.annotate(f"↑ {label_pico}", xy=(month_pico, max_val),
                xytext=(month_pico + 0.2, max_val + 60),
                fontsize=9, color=GOLD, fontweight="bold",
                arrowprops=dict(arrowstyle="->", color=GOLD, lw=1.2))

ax.set_xticks(range(1, 13))
ax.set_xticklabels(meses_pt)
ax.set_ylabel("Número de Pedidos Entregues")
ax.set_xlabel("")
ax.set_title("Picos em Março, Maio e Novembro • 2018 supera 2017 em todos os meses",
             fontsize=10, color="#555")
ax.legend(title="Ano", fontsize=10)

salvar("g07_sazonalidade_mensal_comparativo")

### MEIOS DE PAGAMENTO

In [ ]:
print("Gerando Gráfico 8 – Meios de Pagamento…")

pay_type = (payments.groupby("payment_type")
            .agg(n=("order_id","count"),
                 total=("payment_value","sum"))
            .reset_index())
pay_type = pay_type[pay_type["payment_type"] != "not_defined"]
pay_type["pct_vol"]  = pay_type["n"]     / pay_type["n"].sum()     * 100
pay_type["pct_val"]  = pay_type["total"] / pay_type["total"].sum() * 100

labels_map = {
    "credit_card":"Cartão de Crédito",
    "boleto":     "Boleto Bancário",
    "voucher":    "Voucher",
    "debit_card": "Cartão de Débito",
}
pay_type["label"] = pay_type["payment_type"].map(labels_map).fillna(pay_type["payment_type"])
pay_type = pay_type.sort_values("n", ascending=False)

colors_pay = [C_SUDESTE, ACCENT, GOLD, C_NORTE, C_CO]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Meios de Pagamento – Volume e Valor", fontsize=15,
             fontweight="bold", y=1.01)

# Volume de pedidos
ax = axes[0]
wedges, texts, autotexts = ax.pie(
    pay_type["pct_vol"],
    labels=None,
    colors=colors_pay[:len(pay_type)],
    autopct="%1.1f%%",
    startangle=90,
    pctdistance=0.75,
    wedgeprops=dict(width=0.50, edgecolor="white", linewidth=2.5)
)
for at in autotexts:
    at.set_fontsize(11); at.set_fontweight("bold"); at.set_color("white")
ax.legend(pay_type["label"], loc="lower center",
          bbox_to_anchor=(0.5, -0.12), fontsize=9, ncol=2)
ax.set_title(f"% de Pedidos por Método\n"
             f"(Total: {pay_type['n'].sum():,} transações)")

# Barras – valor total movimentado
ax2 = axes[1]
bars = ax2.barh(pay_type["label"], pay_type["total"] / 1e6,
                color=colors_pay[:len(pay_type)], edgecolor="white", height=0.5)
for bar, (_, row) in zip(bars, pay_type.iterrows()):
    ax2.text(bar.get_width() + 0.2,
             bar.get_y() + bar.get_height()/2,
             f"R$ {row['total']/1e6:.1f}M  ({row['pct_val']:.1f}%)",
             va="center", fontsize=9.5)
ax2.set_xlabel("Valor Total Movimentado (R$ milhões)")
ax2.set_title("Faturamento por Método de Pagamento\n"
              "→ Crédito representa a maior parcela do valor")
ax2.set_xlim(0, pay_type["total"].max() / 1e6 * 1.45)

salvar("g08_meios_pagamento")

### DISTRIBUIÇÃO DE PARCELAS

In [ ]:
print("Gerando Gráfico 9 – Parcelas…")

inst = (payments[payments["payment_type"] == "credit_card"]
        ["payment_installments"]
        .value_counts()
        .reset_index()
        .rename(columns={"index":"parcelas","payment_installments":"n"}))
inst.columns = ["parcelas","n"]
inst = inst[inst["parcelas"] <= 24].sort_values("parcelas")
inst["pct"] = inst["n"] / inst["n"].sum() * 100
inst["acumulado"] = inst["pct"].cumsum()

fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle("Distribuição de Parcelas – Pagamentos com Cartão de Crédito",
             fontsize=15, fontweight="bold", y=1.01)

# Cores: até 3x = verde, 4-10x = amarelo, 10+ = vermelho
def cor_parcela(p):
    if p <= 3:   return C_NORTE
    elif p <= 10: return GOLD
    else:         return ACCENT

bar_cols = [cor_parcela(p) for p in inst["parcelas"]]
bars = ax.bar(inst["parcelas"].astype(str), inst["pct"],
              color=bar_cols, edgecolor="white", width=0.7)

# Linha acumulada
ax2 = ax.twinx()
ax2.plot(range(len(inst)), inst["acumulado"],
         "o--", color=C_SUDESTE, linewidth=2, markersize=5, label="% Acumulado")
ax2.set_ylabel("% Acumulado", color=C_SUDESTE)
ax2.tick_params(axis="y", labelcolor=C_SUDESTE)
ax2.set_ylim(0, 108)

# Linha 80% acumulado
p80_idx = inst[inst["acumulado"] <= 80].index
if len(p80_idx):
    pos80 = inst.index.get_loc(p80_idx[-1])
    ax2.axhline(80, color="#ADB5BD", linestyle=":", linewidth=1.2)
    ax2.annotate("80%", xy=(pos80, 80), xytext=(pos80+1, 83),
                 fontsize=9, color="#555")

for bar in bars:
    v = bar.get_height()
    if v > 1:
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.3,
                f"{v:.1f}%", ha="center", va="bottom", fontsize=8)
ax.set_xlabel("Número de Parcelas")
ax.set_ylabel("% dos Pedidos a Crédito")
ax.set_title("Pagamento à vista ou em até 3x domina • +10x é minoria\n"
             "→ receita financeira em parcelamentos longos é nicho estratégico",
             fontsize=10, color="#555")

p1 = mpatches.Patch(color=C_NORTE, label="1–3x (baixo risco)")
p2 = mpatches.Patch(color=GOLD,    label="4–10x (moderado)")
p3 = mpatches.Patch(color=ACCENT,  label="+10x (alto parcelamento)")
ax.legend(handles=[p1,p2,p3], loc="upper right", fontsize=9)

salvar("g09_distribuicao_parcelas")

### AVALIAÇÕES DOS CLIENTES (REVIEW SCORE)

In [ ]:
print("Gerando Gráfico 10 – Avaliações dos Clientes…")

score_counts = (reviews["review_score"].value_counts()
                .sort_index().reset_index())
score_counts.columns = ["nota","n"]
score_counts["pct"] = score_counts["n"] / score_counts["n"].sum() * 100
score_counts["acumulado_pos"] = score_counts.sort_values("nota", ascending=False)["pct"].cumsum().values[::-1]
nota_media = reviews["review_score"].mean()

star_colors = {1:"#C0392B",2:"#E74C3C",3:"#F39C12",4:"#2ECC71",5:"#27AE60"}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Satisfação dos Clientes – Distribuição de Avaliações",
             fontsize=15, fontweight="bold", y=1.01)

ax = axes[0]
bars = ax.bar(["★"*int(n) for n in score_counts["nota"]],
              score_counts["pct"],
              color=[star_colors[int(n)] for n in score_counts["nota"]],
              edgecolor="white", width=0.6)
for bar, (_, row) in zip(bars, score_counts.iterrows()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f"{row['pct']:.1f}%\n({row['n']:,})",
            ha="center", va="bottom", fontsize=9.5, fontweight="bold")
ax.set_ylabel("% dos Pedidos")
ax.set_xlabel("Nota de Avaliação")
ax.set_title(f"Nota Média Global: {nota_media:.2f} ★\n"
             f"({score_counts[score_counts['nota']>=4]['pct'].sum():.0f}% dos clientes deram 4 ou 5 estrelas)")
ax.set_ylim(0, score_counts["pct"].max() * 1.22)

# NPS simplificado (Promotores 4-5 vs Detratores 1-2 vs Neutros 3)
ax2 = axes[1]
categorias = {
    "Detratores\n(★1–2)":  score_counts[score_counts["nota"]<=2]["pct"].sum(),
    "Neutros\n(★3)":       score_counts[score_counts["nota"]==3]["pct"].sum(),
    "Promotores\n(★4–5)":  score_counts[score_counts["nota"]>=4]["pct"].sum(),
}
cores_nps = [ACCENT, GOLD, C_NORTE]
bars2 = ax2.bar(list(categorias.keys()), list(categorias.values()),
                color=cores_nps, edgecolor="white", width=0.55)
nps = categorias["Promotores\n(★4–5)"] - categorias["Detratores\n(★1–2)"]
for bar in bars2:
    v = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.4,
             f"{v:.1f}%", ha="center", va="bottom",
             fontsize=13, fontweight="bold")
ax2.set_ylabel("% dos Clientes")
ax2.set_title(f"Índice de Satisfação (estilo NPS)\n"
              f"Promotores − Detratores = {nps:.1f} pontos")
ax2.set_ylim(0, max(categorias.values()) * 1.2)

salvar("g10_avaliacoes_clientes")

## SEÇÃO 2 – DESEMPENHO DE PRODUTOS

### FATURAMENTO TOTAL POR CATEGORIA (3 ANOS)

In [ ]:
print("Gerando Gráfico 11 – Faturamento por Categoria…")

fat_cat = (items_full[items_full["order_status"] == "delivered"]
           .groupby("category_en")
           .agg(receita=("price","sum"),
                n_pedidos=("order_id","nunique"))
           .reset_index()
           .dropna()
           .sort_values("receita", ascending=True)
           .tail(20))
fat_cat["ticket_medio"] = fat_cat["receita"] / fat_cat["n_pedidos"]
fat_total = fat_cat["receita"].sum()
fat_cat["pct"] = fat_cat["receita"] / items_full["price"].sum() * 100

norm_f = Normalize(fat_cat["receita"].min(), fat_cat["receita"].max())
cmap_f = LinearSegmentedColormap.from_list("fat", ["#85C1E9","#1A5276"])
colors_f = [cmap_f(norm_f(v)) for v in fat_cat["receita"]]

fig, ax = plt.subplots(figsize=(14, 10))
fig.suptitle("Faturamento Total por Categoria de Produto (2016–2018)",
             fontsize=15, fontweight="bold", y=1.01)

bars = ax.barh(fat_cat["category_en"], fat_cat["receita"] / 1e6,
               color=colors_f, edgecolor="white", height=0.72)
for bar, (_, row) in zip(bars, fat_cat.iterrows()):
    ax.text(bar.get_width() + 0.05,
            bar.get_y() + bar.get_height()/2,
            f"R$ {row['receita']/1e6:.2f}M  |  {row['n_pedidos']:,} ped.  |  TM R$ {row['ticket_medio']:.0f}",
            va="center", fontsize=8)
ax.set_xlabel("Faturamento Total (R$ milhões)")
ax.set_title("Cama/Banho lidera em volume, mas Relógios e Beleza têm alto ticket\n"
             "→ diversificar para categorias de maior valor agrega receita com menos pedidos",
             fontsize=10, color="#555")
ax.set_xlim(0, fat_cat["receita"].max() / 1e6 * 1.55)

sm = cm.ScalarMappable(cmap=cmap_f, norm=norm_f)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.35, pad=0.01)
cbar.set_label("Receita (R$)", fontsize=9)

salvar("g11_faturamento_categoria_total")

### FATURAMENTO POR CATEGORIA × ANO (evolução temporal)

In [ ]:
print("Gerando Gráfico 12 – Faturamento por Categoria × Ano…")

fat_year = (items_full[items_full["order_status"] == "delivered"]
            .groupby(["category_en","year"])["price"].sum()
            .reset_index(name="receita")
            .dropna())

# Top 10 categorias por receita total
top10_cats = (fat_year.groupby("category_en")["receita"].sum()
              .nlargest(10).index.tolist())
fat_year_t = fat_year[fat_year["category_en"].isin(top10_cats)].copy()
fat_year_piv = fat_year_t.pivot(index="category_en", columns="year", values="receita").fillna(0)
fat_year_piv = fat_year_piv.sort_values(fat_year_piv.columns[-1], ascending=True)

# Crescimento 2017→2018
fat_year_piv["growth_18vs17"] = (
    (fat_year_piv.get(2018,0) - fat_year_piv.get(2017,0))
    / fat_year_piv.get(2017,1) * 100)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle("Evolução do Faturamento por Categoria (Top 10)", fontsize=15,
             fontweight="bold", y=1.01)

# Barras agrupadas
ax = axes[0]
years = [c for c in fat_year_piv.columns if isinstance(c, int)]
n_years = len(years)
bar_width = 0.22
year_colors = {2016:"#AED6F1", 2017:C_SUDESTE, 2018:ACCENT}
x = np.arange(len(fat_year_piv))
for i, yr in enumerate(years):
    offset = (i - n_years/2 + 0.5) * bar_width
    bars = ax.barh(x + offset, fat_year_piv[yr] / 1e6,
                   height=bar_width, color=year_colors.get(yr,"#95A5A6"),
                   edgecolor="white", label=str(yr))
ax.set_yticks(x)
ax.set_yticklabels(fat_year_piv.index, fontsize=9)
ax.set_xlabel("Faturamento (R$ milhões)")
ax.set_title("Comparativo Anual por Categoria\n(Top 10 em receita)")
ax.legend(title="Ano", fontsize=9)

# Crescimento 2017→2018
ax2 = axes[1]
growth = fat_year_piv["growth_18vs17"].sort_values()
bar_g_colors = [C_NORTE if v >= 0 else ACCENT for v in growth]
bars2 = ax2.barh(growth.index, growth, color=bar_g_colors, edgecolor="white", height=0.55)
ax2.axvline(0, color="black", linewidth=1.0)
for bar in bars2:
    v = bar.get_width()
    x_pos = v + 1 if v >= 0 else v - 1
    align = "left" if v >= 0 else "right"
    ax2.text(x_pos, bar.get_y() + bar.get_height()/2,
             f"{v:+.1f}%", va="center", ha=align, fontsize=9)
ax2.set_xlabel("Crescimento de Faturamento 2017 → 2018 (%)")
ax2.set_title("Taxa de Crescimento por Categoria\n(2017 → 2018)")
p_pos = mpatches.Patch(color=C_NORTE, label="Crescimento")
p_neg = mpatches.Patch(color=ACCENT,  label="Queda")
ax2.legend(handles=[p_pos,p_neg], fontsize=9)

salvar("g12_faturamento_categoria_ano")

### CONCENTRAÇÃO DE FATURAMENTO POR ESTADO × ANO

In [ ]:
print("Gerando Gráfico 13 – Concentração de Faturamento por Estado × Ano…")

fat_state_year = (items_full[items_full["order_status"] == "delivered"]
                  .groupby(["customer_state","year"])["price"]
                  .sum().reset_index(name="receita"))
fat_state_year["region"] = fat_state_year["customer_state"].map(STATE_REGION)

# Top 15 estados por receita total
top15_states = (fat_state_year.groupby("customer_state")["receita"].sum()
                .nlargest(15).index.tolist())
fat_state_f = fat_state_year[fat_state_year["customer_state"].isin(top15_states)]
fat_piv = fat_state_f.pivot(index="customer_state", columns="year", values="receita").fillna(0)
fat_piv = fat_piv.loc[fat_piv.sum(axis=1).sort_values(ascending=True).index]
fat_piv_pct = fat_piv.div(fat_piv.sum()) * 100   # share anual

years_av = [c for c in fat_piv.columns if isinstance(c, int)]
year_colors2 = {2016:"#AED6F1", 2017:C_SUDESTE, 2018:ACCENT}

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
fig.suptitle("Concentração de Faturamento por Estado (Top 15)", fontsize=15,
             fontweight="bold", y=1.01)

# Barras empilhadas – receita absoluta
ax = axes[0]
bottom = np.zeros(len(fat_piv))
for yr in years_av:
    ax.barh(fat_piv.index, fat_piv[yr] / 1e6, left=bottom,
            color=year_colors2.get(yr,"#95A5A6"), edgecolor="white",
            height=0.72, label=str(yr))
    bottom += fat_piv[yr].values / 1e6
ax.set_xlabel("Faturamento Total (R$ milhões)")
ax.set_title("Faturamento Absoluto Empilhado por Ano\n"
             "(SP, RJ e MG concentram > 60% da receita)")
ax.legend(title="Ano", fontsize=9)
for i, state in enumerate(fat_piv.index):
    total = fat_piv.loc[state].sum() / 1e6
    region = STATE_REGION.get(state,"")
    ax.text(total + 0.3, i, f"R$ {total:.1f}M", va="center", fontsize=8.5)

# Heatmap – share por estado/ano
ax2 = axes[1]
sns.heatmap(fat_piv_pct[years_av].T,
            ax=ax2, cmap="Blues", annot=True, fmt=".1f",
            linewidths=0.5, linecolor="white",
            cbar_kws={"label":"Share (% do faturamento anual)"},
            annot_kws={"size": 8})
ax2.set_xlabel("")
ax2.set_ylabel("Ano")
ax2.set_title("Share de Faturamento por Estado × Ano\n"
              "(% dentro de cada ano – SP lidera consistentemente)")
ax2.set_yticklabels(years_av, rotation=0)

salvar("g13_concentracao_faturamento_estado_ano")

### TEMPO DE ENTREGA POR ESTADO

In [ ]:
print("Gerando Gráfico 14 – Tempo de Entrega por Estado…")

delivered = orders_full[
    orders_full["order_delivered_customer_date"].notna() &
    orders_full["order_status"].eq("delivered")
].copy()
delivered["total_days"] = (
    (delivered["order_delivered_customer_date"] -
     delivered["order_purchase_timestamp"]).dt.total_seconds() / 86400)
delivered["estimated_days"] = (
    (delivered["order_estimated_delivery_date"] -
     delivered["order_purchase_timestamp"]).dt.total_seconds() / 86400)
delivered["diff_days"] = delivered["total_days"] - delivered["estimated_days"]
delivered["on_time"]   = delivered["diff_days"] <= 0

state_delivery = (delivered.groupby("customer_state")
                  .agg(
                      avg_days    = ("total_days",  "mean"),
                      median_days = ("total_days",  "median"),
                      on_time_pct = ("on_time",     lambda x: x.mean() * 100),
                      avg_review  = ("order_id",
                                     lambda x: reviews.set_index("order_id")
                                     .loc[reviews.set_index("order_id").index.isin(x),
                                          "review_score"].mean()
                                     if not x.empty else np.nan)
                  )
                  .reset_index()
                  .sort_values("avg_days", ascending=True))

# Review por estado
rev_state = (reviews[["order_id","review_score"]]
             .drop_duplicates("order_id")
             .merge(orders_full[["order_id","customer_state"]], on="order_id", how="left")
             .groupby("customer_state")["review_score"].mean().reset_index())
state_delivery = state_delivery.merge(rev_state, on="customer_state", how="left")

norm_d = Normalize(state_delivery["avg_days"].min(), state_delivery["avg_days"].max())
cmap_d = LinearSegmentedColormap.from_list("days", ["#27AE60","#F39C12","#E74C3C"])
colors_d = [cmap_d(norm_d(v)) for v in state_delivery["avg_days"]]

fig, axes = plt.subplots(1, 2, figsize=(18, 10))
fig.suptitle("Tempo de Entrega e Avaliação por Estado", fontsize=15,
             fontweight="bold", y=1.01)

# Tempo médio de entrega
ax = axes[0]
bars = ax.barh(state_delivery["customer_state"],
               state_delivery["avg_days"],
               color=colors_d, edgecolor="white", height=0.72)
media_dias = state_delivery["avg_days"].mean()
ax.axvline(media_dias, color=C_SUDESTE, linestyle="--", linewidth=1.5,
           label=f"Média: {media_dias:.1f} dias")
for bar, (_, row) in zip(bars, state_delivery.iterrows()):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f"{row['avg_days']:.1f}d  |  ✓{row['on_time_pct']:.0f}%",
            va="center", fontsize=8)
ax.set_xlabel("Tempo Médio de Entrega (dias)")
ax.set_title("Prazo Médio por Estado\n"
             "(verde = rápido | vermelho = lento | % = entregas no prazo)")
ax.legend(fontsize=9)

sm = cm.ScalarMappable(cmap=cmap_d, norm=norm_d)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, shrink=0.35, pad=0.02)
cbar.set_label("Dias", fontsize=9)

# Entrega × Avaliação (scatter com tendência)
ax2 = axes[1]
sc = ax2.scatter(state_delivery["avg_days"],
                 state_delivery["review_score"],
                 c=state_delivery["avg_days"], cmap=cmap_d, norm=norm_d,
                 s=100, zorder=5, edgecolors="white", linewidths=0.8)
for _, row in state_delivery.iterrows():
    ax2.annotate(row["customer_state"],
                 (row["avg_days"], row["review_score"]),
                 textcoords="offset points", xytext=(5,3), fontsize=7.5)
# Linha de tendência
x_r = state_delivery["avg_days"].values
y_r = state_delivery["review_score"].values
mask = ~np.isnan(y_r)
slope, intercept, r, p, _ = stats.linregress(x_r[mask], y_r[mask])
x_line = np.linspace(x_r.min(), x_r.max(), 100)
ax2.plot(x_line, slope * x_line + intercept, "--", color=C_SUDESTE,
         linewidth=1.5, label=f"Tendência (r={r:.2f})")
ax2.set_xlabel("Tempo Médio de Entrega (dias)")
ax2.set_ylabel("Nota Média de Avaliação")
ax2.set_title(f"Tempo de Entrega vs Satisfação por Estado\n"
              f"Correlação fraca (r={r:.2f}) → produto importa mais que prazo")
ax2.legend(fontsize=9)
ax2.set_ylim(3, 5)

salvar("g14_entrega_avaliacao_estado")

### PAINEL RESUMO EXECUTIVO

In [ ]:
print("Gerando Gráfico 15 – Painel Resumo…")

# Coleta de KPIs finais
total_clientes = customers["customer_unique_id"].nunique()
total_pedidos  = orders[orders["order_status"]=="delivered"]["order_id"].nunique()
receita_total  = items[["order_id","price"]].merge(
    orders[orders["order_status"]=="delivered"][["order_id"]], on="order_id")["price"].sum()
ticket_medio   = receita_total / total_pedidos
nota_media_kpi = reviews["review_score"].mean()
on_time_kpi    = delivered["on_time"].mean() * 100
taxa_recompra  = (purchase_counts[purchase_counts["n_compras"] > 1].shape[0]
                  / purchase_counts.shape[0] * 100)
vendedores_kpi = sellers["seller_id"].nunique()

kpis = [
    ("💰", "Receita Total",      f"R$ {receita_total/1e6:.1f}M",  "#27AE60"),
    ("📦", "Pedidos Entregues",  f"{total_pedidos:,}",             "#3498DB"),
    ("👥", "Clientes Únicos",    f"{total_clientes:,}",            "#2ECC71"),
    ("🛍️", "Ticket Médio",      f"R$ {ticket_medio:.0f}",         "#F39C12"),
    ("⭐", "Nota Média",         f"{nota_media_kpi:.2f} / 5,0",    "#E74C3C"),
    ("🚚", "Entrega no Prazo",   f"{on_time_kpi:.1f}%",           "#1ABC9C"),
    ("🏪", "Vendedores Ativos",  f"{vendedores_kpi:,}",            "#8E44AD"),
    ("🔄", "Taxa de Recompra",   f"{taxa_recompra:.1f}%",          "#E67E22"),
]

fig = plt.figure(figsize=(18, 9))
fig.patch.set_facecolor("#1C2833")
gs_kpi = GridSpec(2, 4, figure=fig, hspace=0.35, wspace=0.25,
                  top=0.82, bottom=0.05, left=0.02, right=0.98)

for i, (icone, label, valor, cor) in enumerate(kpis):
    row, col = divmod(i, 4)
    ax_k = fig.add_subplot(gs_kpi[row, col])
    ax_k.set_facecolor(cor)
    ax_k.set_xlim(0, 1); ax_k.set_ylim(0, 1); ax_k.axis("off")
    ax_k.text(0.5, 0.70, valor, ha="center", va="center",
              fontsize=20, fontweight="bold", color="white",
              transform=ax_k.transAxes)
    ax_k.text(0.5, 0.28, f"{icone}  {label}", ha="center", va="center",
              fontsize=9.5, color="white", alpha=0.9,
              transform=ax_k.transAxes)

fig.text(0.5, 0.94, "OLIST – Painel Executivo de Performance",
         ha="center", fontsize=18, fontweight="bold", color="white")
fig.text(0.5, 0.89, "Período: 2016–2018  |  Fonte: Olist Dataset",
         ha="center", fontsize=10, color="#BDC3C7")

salvar("g15_painel_executivo_kpis")